## **Practice 1°: N-gram** - *3° Partial*
* January 08°, 2026
#### ESCOM - IPN: *Natural Language Processing*
#### Prof. Marco Antonio

#### *B.S. in Data Science* - 6AV1
> Sánchez García Miguel Alexander

#### **0° N-gram**

**a.** Get the text

In [143]:
def extract_text_from_pdf(file_path, start_page=1):
    try:
        import PyPDF2
    except ImportError:
        raise ImportError("PyPDF2 no está instalado. Instálalo con: pip install PyPDF2")
    
    text_pages = []
    with open(file_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        # Extraer texto desde start_page hasta el final
        for page_num in range(start_page, len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text_pages.append(page.extract_text() or "")
    
    return "\n".join(text_pages)

In [144]:
text = extract_text_from_pdf('Data/Docs/constitucion.pdf', start_page=1)

print(text[:500])

 
 CONSTITUCIÓN POLÍTICA DE LOS ESTADOS UNIDOS MEXICANOS  
 
CÁMARA DE DIPUTADOS DEL H. CONGRESO DE LA UNIÓN 
Secretaría General  
Secretaría de Servicios Parlamentarios  Última Reforma DOF 15-10-2025  
 
 
2 de 402 Está prohibida la esclavitud en los Estados Unidos Mexicanos. Los esclavos del extranjero que  entren 
al territorio nacional alcanzarán, por este solo hecho, su libertad y la protección de las leyes.  
 
Queda prohibida toda discriminación motivada por origen étnico o nacional, el g


**b.** Preprocess the text

In [145]:
import spacy

nlp = spacy.load(
    "es_core_news_sm",
    disable=["parser", "ner", "tagger"]
)

def tokenize_large_text(text, chunk_size=300_000):
    tokens = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        doc = nlp(chunk)
        tokens.extend([t.text.lower() for t in doc if not t.is_punct and not t.is_space])
    return tokens

tokens = tokenize_large_text(text)

print(tokens[:50])

['constitución', 'política', 'de', 'los', 'estados', 'unidos', 'mexicanos', 'cámara', 'de', 'diputados', 'del', 'h.', 'congreso', 'de', 'la', 'unión', 'secretaría', 'general', 'secretaría', 'de', 'servicios', 'parlamentarios', 'última', 'reforma', 'dof', '15-10-2025', '2', 'de', '402', 'está', 'prohibida', 'la', 'esclavitud', 'en', 'los', 'estados', 'unidos', 'mexicanos', 'los', 'esclavos', 'del', 'extranjero', 'que', 'entren', 'al', 'territorio', 'nacional', 'alcanzarán', 'por', 'este']


**c.** Perfom the n-grams

In [146]:
from collections import defaultdict, Counter

unigrams = Counter(tokens)
bigrams = defaultdict(Counter)

for i in range(len(tokens)-1):
    w1, w2 = tokens[i], tokens[i+1]
    bigrams[w1][w2] += 1

vocab = set(tokens)
V = len(vocab)

In [147]:
trigrams = defaultdict(Counter)

for i in range(len(tokens) - 2):
    context = (tokens[i], tokens[i+1])
    next_word = tokens[i+2]
    trigrams[context][next_word] += 1

In [148]:
'''def predict_next_word_trigram(w1, w2, trigrams, tokens):
     context = (w1.lower(), w2.lower())

     # Known context
     if context in trigrams:
         return trigrams[context].most_common(1)[0][0]

     # Backoff to bigram
     for (c1, c2), counter in trigrams.items():
         if c2 == w2.lower():
             return counter.most_common(1)[0][0]

     # Most common word
     return Counter(tokens).most_common(1)[0][0]'''

'def predict_next_word_trigram(w1, w2, trigrams, tokens):\n     context = (w1.lower(), w2.lower())\n\n     # Known context\n     if context in trigrams:\n         return trigrams[context].most_common(1)[0][0]\n\n     # Backoff to bigram\n     for (c1, c2), counter in trigrams.items():\n         if c2 == w2.lower():\n             return counter.most_common(1)[0][0]\n\n     # Most common word\n     return Counter(tokens).most_common(1)[0][0]'

**d.** Function to predict the next word using backoff

In [149]:
def predict_next_word(w1, w2):
    w1 = w1.lower()
    w2 = w2.lower()
    context = (w1, w2)

    # Trigram 
    if context in trigrams:
        for word, count in trigrams[context].most_common():
            if word != w1 and word != w2:
                return word
    
    # Backoff to bigram 
    if w2 in bigrams:
        for word, count in bigrams[w2].most_common():
            if word != w1 and word != w2:
                return word
    
    # Backoff to unigram
    for word, count in unigrams.most_common():
        if word != w1 and word != w2:
            return word
    
    return unigrams.most_common(1)[0][0]

**e.** Evaluate the model

In [150]:
correct = 0
total = 0

for i in range(len(tokens) - 2):
    w1, w2 = tokens[i], tokens[i+1]
    real = tokens[i+2]
    pred = predict_next_word(w1, w2)
    if pred == real:
        correct += 1
    total += 1

accuracy = correct / total
print("Accuracy:", accuracy)

Accuracy: 0.6284164810937171


**f**. The user evaluate the predictions

In [170]:
word = input("Enter one word: ").strip()

predicted_word = predict_next_word(word, word)

print("Predicted next word:", predicted_word)


Predicted next word: unico
